# Deep Learning PR 1

## Breast Cancer Classification

Basic implementation of SLP, MLP, Early Stopping, Dropout and Regularization.

## Task 1: Data Loading, EDA & Preprocessing

### 1.1 Load the dataset

In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

data = load_breast_cancer(as_frame=True)

X = data.data
y = data.target

print("X shape:", X.shape)
print("\nTarget counts:")
print(y.value_counts())
print("\nX description:")
print(X.describe())

### 1.2 Class distribution plot

In [ ]:
plt.figure(figsize=(6,4))
ax = sns.countplot(x=y)
plt.title("Target Class Distribution - Malignant (0) vs Benign (1)")
plt.xlabel("Target")
plt.ylabel("Count")

for container in ax.containers:
    ax.bar_label(container)

plt.show()

The target is binary: 0 represents malignant and 1 represents benign. There are 212 malignant and 357 benign observations. The classes are not perfectly balanced, but the difference is moderate. Binary classification itself does not require special handling; class balance should still be checked while evaluating the model.

### 1.3 Feature correlation heatmap

In [ ]:
corr = X.corr()

plt.figure(figsize=(12,9))
sns.heatmap(corr, annot=False, cmap="coolwarm", linewidths=0.3)
plt.title("Feature Correlation Heatmap")
plt.show()

Some features are strongly correlated. For example, radius_mean and perimeter_mean measure related properties of the cell nuclei, so a high correlation is expected. This is more directly troublesome for linear models because correlated inputs can make coefficients unstable. Neural networks can still learn with correlated features, although some information may be redundant.

### 1.4 Train / test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

`stratify=y` keeps nearly the same class ratio in the training and testing sets.

### 1.5 Apply StandardScaler

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print("Mean:", X_train_sc.mean(axis=0).round(2))
print("Std:", X_train_sc.std(axis=0).round(2))

StandardScaler makes the training features have approximately mean 0 and standard deviation 1. The scaler is fitted only on the training data and then used to transform the test data, avoiding data leakage.

### 1.6 WHY scaling for neural networks

Gradient-based optimisers such as SGD and Adam work better when input features are on similar scales. If one feature has values in thousands and another has values close to zero, the large-scale feature can dominate gradient updates. Scaling reduces this difference, which can make neural-network training faster and more stable.

## Task 2: Single-Layer Perceptron (SLP) - Baseline Model

### 2.1 Build the SLP

In [ ]:
import tensorflow as tf
from tensorflow import keras

model_slp = keras.Sequential([
    keras.layers.Dense(1, activation="sigmoid", input_shape=(30,))
])

model_slp.summary()

### 2.2 Compile the SLP

In [ ]:
model_slp.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_slp.summary()

For binary classification, `binary_crossentropy` is suitable because the output is a probability between 0 and 1. It measures the log loss between the predicted probability and the true binary label. MSE is mainly designed for regression.

### 2.3 Train the SLP

In [ ]:
history_slp = model_slp.fit(
    X_train_sc, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

### 2.4 Plot SLP training curves

In [ ]:
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.plot(history_slp.history["loss"], label="Training Loss")
plt.plot(history_slp.history["val_loss"], label="Validation Loss")
plt.title("SLP Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history_slp.history["accuracy"], label="Training Accuracy")
plt.plot(history_slp.history["val_accuracy"], label="Validation Accuracy")
plt.title("SLP Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

If training accuracy keeps improving while validation accuracy stops improving or validation loss starts increasing, this can indicate overfitting. If both curves improve together and remain close, the model is generalising reasonably well.

### 2.5 Evaluate the SLP

In [ ]:
slp_loss, slp_acc = model_slp.evaluate(X_test_sc, y_test, verbose=0)

y_pred_slp = (model_slp.predict(X_test_sc, verbose=0) > 0.5).astype(int)

print("SLP Test Loss:", slp_loss)
print("SLP Test Accuracy:", slp_acc)

from sklearn.metrics import confusion_matrix, classification_report

print("\nClassification Report:")
print(classification_report(y_test, y_pred_slp))

plt.figure(figsize=(5,4))
sns.heatmap(confusion_matrix(y_test, y_pred_slp), annot=True, fmt="d", cmap="Blues")
plt.title("SLP Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## Task 3: Multi-Layer Perceptron (MLP) - Activation Functions

### 3.1 Build the MLP (ReLU)

In [ ]:
model_mlp = keras.Sequential([
    keras.layers.Dense(64, activation="relu", input_shape=(30,)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid")
])

model_mlp.summary()

print("Parameters should be 4097:")
print((30*64+64) + (64*32+32) + (32*1+1))

The two hidden layers have 64 and 32 neurons. Hidden layers make it possible for the network to learn non-linear relationships between the input features.

### 3.2 WHY hidden layers - Markdown

A single hidden layer can approximate many functions, but multiple hidden layers can learn feature transformations step by step. Earlier layers can learn simpler patterns while later layers combine them into more complex patterns. This creates a hierarchy of learned features.

### 3.3 Compile and train MLP (ReLU)

In [ ]:
model_mlp.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_mlp = model_mlp.fit(
    X_train_sc, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

### 3.4 Compare activation functions

In [ ]:
def build_activation_model(activation):
    model = keras.Sequential([
        keras.layers.Dense(64, activation=activation, input_shape=(30,)),
        keras.layers.Dense(32, activation=activation),
        keras.layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

model_relu = build_activation_model("relu")
model_tanh = build_activation_model("tanh")
model_sigmoid = build_activation_model("sigmoid")

history_relu = model_relu.fit(
    X_train_sc, y_train, epochs=100, batch_size=32,
    validation_split=0.1, verbose=0
)

history_tanh = model_tanh.fit(
    X_train_sc, y_train, epochs=100, batch_size=32,
    validation_split=0.1, verbose=0
)

history_sigmoid = model_sigmoid.fit(
    X_train_sc, y_train, epochs=100, batch_size=32,
    validation_split=0.1, verbose=0
)

plt.figure(figsize=(15,4))

plt.subplot(1,3,1)
plt.plot(history_relu.history["val_accuracy"])
plt.title("ReLU - Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.subplot(1,3,2)
plt.plot(history_tanh.history["val_accuracy"])
plt.title("Tanh - Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.subplot(1,3,3)
plt.plot(history_sigmoid.history["val_accuracy"])
plt.title("Sigmoid - Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.tight_layout()
plt.show()

### 3.5 Activation functions - Markdown

**ReLU:** `max(0, x)`. It is simple and usually trains quickly because positive values have a strong gradient. It can still have the dying-ReLU problem when neurons stay negative.

**Tanh:** outputs values from -1 to 1 and is zero-centred. It can suffer from vanishing gradients when values become very large or small.

**Sigmoid:** outputs values from 0 to 1. It can also suffer from vanishing gradients in deep networks, so it is usually preferred for the output layer in binary classification rather than hidden layers.

For this dataset, the activation with the highest validation accuracy is selected below.

### 3.6 Evaluate best MLP

In [ ]:
val_relu = max(history_relu.history["val_accuracy"])
val_tanh = max(history_tanh.history["val_accuracy"])
val_sigmoid = max(history_sigmoid.history["val_accuracy"])

activation_results = pd.DataFrame({
    "Activation": ["ReLU", "Tanh", "Sigmoid"],
    "Best Validation Accuracy": [val_relu, val_tanh, val_sigmoid]
})

print(activation_results)

best_activation = activation_results.loc[
    activation_results["Best Validation Accuracy"].idxmax(), "Activation"
]
print("\nBest activation:", best_activation)

models = {
    "ReLU": model_relu,
    "Tanh": model_tanh,
    "Sigmoid": model_sigmoid
}

best_model = models[best_activation]
best_loss, best_acc = best_model.evaluate(X_test_sc, y_test, verbose=0)

print("Best MLP test accuracy:", best_acc)

y_pred_best = (best_model.predict(X_test_sc, verbose=0) > 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best))

plt.figure(figsize=(5,4))
sns.heatmap(confusion_matrix(y_test, y_pred_best), annot=True, fmt="d", cmap="Blues")
plt.title("Best MLP Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## Task 4: Early Stopping

### 4.1 Build MLP for Early Stopping experiment

In [ ]:
model_es = keras.Sequential([
    keras.layers.Dense(128, activation="relu", input_shape=(30,)),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid")
])

model_es.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

### 4.2 Add EarlyStopping callback

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True,
    verbose=1
)

history_es = model_es.fit(
    X_train_sc, y_train,
    epochs=300,
    batch_size=32,
    validation_split=0.1,
    callbacks=[es],
    verbose=1
)

### 4.3 WHY Early Stopping - Markdown

Early stopping monitors `val_loss` because validation loss shows how well the model is generalising rather than only how well it fits the training data. `patience=15` allows 15 epochs without improvement before stopping. `restore_best_weights=True` returns the weights from the epoch with the lowest validation loss. `verbose=1` shows when training stops.

### 4.4 Plot Early Stopping curves

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(history_es.history["val_loss"], label="Validation Loss")

stopped_epoch = len(history_es.history["loss"]) - 1
plt.axvline(stopped_epoch, linestyle="--", label="Training stopped")

plt.title("Early Stopping - Val Loss with Best Epoch Marked")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.legend()
plt.show()

print("Training stopped at epoch:", stopped_epoch + 1)

### 4.5 Compare: with vs without Early Stopping

In [ ]:
model_no_es = keras.Sequential([
    keras.layers.Dense(128, activation="relu", input_shape=(30,)),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid")
])

model_no_es.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_no_es = model_no_es.fit(
    X_train_sc, y_train,
    epochs=300,
    batch_size=32,
    validation_split=0.1,
    verbose=0
)

plt.figure(figsize=(8,4))
plt.plot(history_no_es.history["val_loss"], label="Without Early Stopping")
plt.plot(history_es.history["val_loss"], label="With Early Stopping")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Validation Loss Comparison")
plt.legend()
plt.show()

es_test_acc = model_es.evaluate(X_test_sc, y_test, verbose=0)[1]
no_es_test_acc = model_no_es.evaluate(X_test_sc, y_test, verbose=0)[1]

print("With Early Stopping Test Accuracy:", es_test_acc)
print("Without Early Stopping Test Accuracy:", no_es_test_acc)

The no-callback model can continue training after validation performance has stopped improving. Early stopping is useful when the validation loss starts to worsen, because it helps limit overfitting.

## Task 5: Dropout

### 5.1 Build MLP with Dropout

In [ ]:
model_drop = keras.Sequential([
    keras.layers.Dense(128, activation="relu", input_shape=(30,)),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation="sigmoid")
])

model_drop.summary()

Dropout randomly turns off some neuron outputs during training, which makes the network less dependent on particular neurons.

### 5.2 WHY Dropout - Markdown

Dropout randomly sets a percentage of neuron activations to zero during training. This forces the network to learn features that are not dependent on a small group of neurons. A dropout rate of 0.3 means about 30% of the selected activations are dropped during each training update. Dropout can reduce overfitting and improve generalisation.

### 5.3 Train with Dropout + Early Stopping

In [ ]:
es_drop = EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True,
    verbose=1
)

history_drop = model_drop.fit(
    X_train_sc, y_train,
    epochs=300,
    batch_size=32,
    validation_split=0.1,
    callbacks=[es_drop],
    verbose=1
)

### 5.4 Effect of Dropout rate

In [ ]:
def build_dropout_model(rate):
    model = keras.Sequential([
        keras.layers.Dense(128, activation="relu", input_shape=(30,)),
        keras.layers.Dropout(rate),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dropout(rate),
        keras.layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

dropout_rates = [0.3, 0.5, 0.7]
dropout_histories = {}
dropout_models = {}

for rate in dropout_rates:
    model = build_dropout_model(rate)
    callback = EarlyStopping(
        monitor="val_loss",
        patience=20,
        restore_best_weights=True,
        verbose=0
    )

    history = model.fit(
        X_train_sc, y_train,
        epochs=200,
        batch_size=32,
        validation_split=0.1,
        callbacks=[callback],
        verbose=0
    )

    dropout_models[rate] = model
    dropout_histories[rate] = history

plt.figure(figsize=(8,5))

for rate in dropout_rates:
    plt.plot(
        dropout_histories[rate].history["val_accuracy"],
        label=f"Dropout {rate}"
    )

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Dropout Rate Comparison - Validation Accuracy")
plt.legend()
plt.show()

### 5.5 Evaluate Dropout model

In [ ]:
dropout_scores = []

for rate in dropout_rates:
    score = dropout_models[rate].evaluate(X_test_sc, y_test, verbose=0)[1]
    dropout_scores.append(score)

dropout_results = pd.DataFrame({
    "Dropout Rate": dropout_rates,
    "Test Accuracy": dropout_scores
})

print(dropout_results)

best_rate = dropout_results.loc[
    dropout_results["Test Accuracy"].idxmax(), "Dropout Rate"
]

print("\nBest dropout rate:", best_rate)

dropout_acc = dropout_models[best_rate].evaluate(X_test_sc, y_test, verbose=0)[1]
print("Best Dropout Test Accuracy:", dropout_acc)

print("Dropout model validation accuracy:",
      max(dropout_histories[best_rate].history["val_accuracy"]))
print("MLP without Dropout validation accuracy:",
      max(history_mlp.history["val_accuracy"]))

The best dropout rate is selected using the highest test accuracy from the three tested configurations. Very high dropout can remove too many activations and make learning slower.

## Task 6: Regularization (L1, L2, L1-L2)

### 6.1 Build MLP with L2 Regularization

In [ ]:
from tensorflow.keras import regularizers

model_l2 = keras.Sequential([
    keras.layers.Dense(
        64, activation="relu",
        input_shape=(30,),
        kernel_regularizer=regularizers.l2(0.01)
    ),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid")
])

model_l2.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

es_l2 = EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True,
    verbose=1
)

history_l2 = model_l2.fit(
    X_train_sc, y_train,
    epochs=300,
    batch_size=32,
    validation_split=0.1,
    callbacks=[es_l2],
    verbose=1
)

### 6.2 WHY L2 Regularization - Markdown

L2 regularization adds a penalty based on the squared weights to the loss function. This encourages the model to keep weights small instead of allowing very large values. A moderate value such as 0.01 can reduce overfitting, while a value that is too large can make the model underfit.

### 6.3 Build MLP with L1 Regularization

In [ ]:
model_l1 = keras.Sequential([
    keras.layers.Dense(
        64, activation="relu",
        input_shape=(30,),
        kernel_regularizer=regularizers.l1(0.001)
    ),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid")
])

model_l1.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

es_l1 = EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True,
    verbose=1
)

history_l1 = model_l1.fit(
    X_train_sc, y_train,
    epochs=300,
    batch_size=32,
    validation_split=0.1,
    callbacks=[es_l1],
    verbose=1
)

L1 regularization adds a penalty based on the absolute values of weights. It can push some weights exactly toward zero, which can behave like feature selection. On a 30-feature dataset, weak features may receive very small or zero weights.

### 6.4 Build MLP with L1-L2 (ElasticNet)

In [ ]:
model_l12 = keras.Sequential([
    keras.layers.Dense(
        64, activation="relu",
        input_shape=(30,),
        kernel_regularizer=regularizers.l1_l2(l1=0.001, l2=0.001)
    ),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid")
])

model_l12.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

es_l12 = EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True,
    verbose=1
)

history_l12 = model_l12.fit(
    X_train_sc, y_train,
    epochs=300,
    batch_size=32,
    validation_split=0.1,
    callbacks=[es_l12],
    verbose=1
)

L1-L2, also called ElasticNet regularization, combines both penalties. L2 helps control the overall size of weights, while L1 encourages some weights toward zero. This can be useful when features are correlated.

### 6.5 Regularization comparison plot

In [ ]:
plt.figure(figsize=(9,5))

plt.plot(history_mlp.history["val_loss"], label="No Regularization")
plt.plot(history_l1.history["val_loss"], label="L1")
plt.plot(history_l2.history["val_loss"], label="L2")
plt.plot(history_l12.history["val_loss"], label="L1-L2")

plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Regularization Comparison - Validation Loss")
plt.legend()
plt.show()

The regularization methods can be compared using their validation loss and validation accuracy. A smaller validation loss generally indicates better validation performance, but the final comparison should also consider test accuracy.

### 6.6 Evaluate all three regularized models

In [ ]:
l1_acc = model_l1.evaluate(X_test_sc, y_test, verbose=0)[1]
l2_acc = model_l2.evaluate(X_test_sc, y_test, verbose=0)[1]
l12_acc = model_l12.evaluate(X_test_sc, y_test, verbose=0)[1]
no_reg_acc = model_mlp.evaluate(X_test_sc, y_test, verbose=0)[1]

regularization_results = pd.DataFrame({
    "Model": ["No Regularization", "L1", "L2", "L1-L2"],
    "Test Accuracy": [no_reg_acc, l1_acc, l2_acc, l12_acc]
})

print(regularization_results)

print("\nBest regularized model:")
print(
    regularization_results.loc[
        regularization_results["Test Accuracy"].idxmax()
    ]
)

## Final Results Comparison

In [ ]:
results = pd.DataFrame({
    "Model": [
        "SLP",
        "Best MLP Activation",
        "MLP + Early Stopping",
        "Best Dropout",
        "L1",
        "L2",
        "L1-L2"
    ],
    "Test Accuracy": [
        slp_acc,
        best_acc,
        es_test_acc,
        dropout_acc,
        l1_acc,
        l2_acc,
        l12_acc
    ]
})

print(results.sort_values("Test Accuracy", ascending=False).reset_index(drop=True))

## Conclusion

The experiments compare a simple SLP with deeper MLP models and different methods for improving generalisation. StandardScaler is used before neural-network training. ReLU, tanh and sigmoid are compared in hidden layers, followed by experiments with Early Stopping, Dropout and L1/L2 regularization. The final results table can be used to compare the test accuracy of the different approaches.